#  LLM From Scratch — Le Preprocessing

Ce notebook accompagne la série **"Construire un LLM from scratch"**.
Il implémente les 4 étapes du preprocessing :

1. **Chargement et nettoyage du corpus** (Wikipedia FR + CulturaX FR)
2. **Tokenisation avec Byte Pair Encoding (BPE)**
3. **Token Embeddings**
4. **Positional Embeddings**

À la fin, on obtient un pipeline complet : **texte brut → vecteurs prêts pour le Transformer**.

---
>  Testé sur Google Colab (GPU T4). RAM recommandée : 12GB minimum.

##  0. Installation des dépendances

In [1]:
!pip install datasets tokenizers torch transformers tqdm -q

In [2]:
import re
import torch
import torch.nn as nn
import numpy as np
from tqdm import tqdm
from collections import Counter, defaultdict
from datasets import load_dataset
from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.trainers import BpeTrainer
from tokenizers.pre_tokenizers import Whitespace
from huggingface_hub import login
import getpass

# Le token est saisi de facon securisee, il n'apparait pas dans le code
hf_token = getpass.getpass('Entre ton token Hugging Face : ')
login(token=hf_token)

print('Authentifie sur Hugging Face')
print(f'PyTorch version : {torch.__version__}')
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device : {device}')


Entre ton token Hugging Face :  ········


Authentifie sur Hugging Face
PyTorch version : 2.10.0+cu128
Device : cuda


---
##  1. Chargement du corpus

On utilise deux sources :
- **Wikipedia français** : texte encyclopédique, propre et structuré
- **CulturaX français** : texte web filtré, plus varié et plus grand

On échantillonne **~1GB** au total pour rester reproductible sur Colab.

In [3]:
# --- Wikipedia Français ---
# On utilise la version Parquet (datasets >= 2.14 ne supporte plus les loading scripts)
print("Chargement de Wikipedia FR...")
wiki_dataset = load_dataset(
    "wikimedia/wikipedia",
    "20231101.fr",
    split="train"
)

# On garde les 200 000 premiers articles
wiki_texts = wiki_dataset.select(range(min(200_000, len(wiki_dataset))))["text"]
print(f"Wikipedia FR : {len(wiki_texts):,} articles charges")

Chargement de Wikipedia FR...


README.md: 0.00B [00:00, ?B/s]

Resolving data files:   0%|          | 0/17 [00:00<?, ?it/s]

20231101.fr/train-00000-of-00017.parquet:   0%|          | 0.00/769M [00:00<?, ?B/s]

20231101.fr/train-00001-of-00017.parquet:   0%|          | 0.00/422M [00:00<?, ?B/s]

...

20231101.fr/train-00015-of-00017.parquet:   0%|          | 0.00/186M [00:00<?, ?B/s]

20231101.fr/train-00016-of-00017.parquet:   0%|          | 0.00/186M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/2564646 [00:00<?, ? examples/s]

Loading dataset shards:   0%|          | 0/17 [00:00<?, ?it/s]

Wikipedia FR : 200,000 articles charges


In [4]:
# --- CulturaX Francais ---
# Necessite un token HF avec acces accepte sur https://huggingface.co/datasets/uonlp/CulturaX
print('Chargement de CulturaX FR...')
cultura_dataset = load_dataset(
    'uonlp/CulturaX',
    'fr',
    split='train',
    streaming=True
)

# On prend 300 000 exemples en streaming
oscar_texts = []
for i, example in enumerate(cultura_dataset):
    oscar_texts.append(example['text'])
    if i >= 299_999:
        break

print(f'CulturaX FR : {len(oscar_texts):,} documents charges')


Chargement de CulturaX FR...


README.md: 0.00B [00:00, ?B/s]

Resolving data files:   0%|          | 0/512 [00:00<?, ?it/s]

CulturaX FR : 300,000 documents charges


In [5]:
# Fusion des deux corpus
all_texts = list(wiki_texts) + oscar_texts
print(f"Corpus total : {len(all_texts):,} documents")

# Aperçu
print("\nExemple Wikipedia :")
print(all_texts[0][:300])
print("\nExemple OSCAR :")
print(all_texts[200_001][:300])

Corpus total : 500,000 documents

Exemple Wikipedia :
Antoine Meillet, né le  à Moulins (Allier) et mort le  à Châteaumeillant (Cher), est un philologue français, le principal linguiste français des premières décennies du .

Biographie

Enfance et formation 
Paul Jules Antoine Meillet est d'origine bourbonnaise, fils d'un notaire de Châteaumeillant (Ch

Exemple OSCAR :
Servir notre prochain, et en particulier les personnes que nous rencontrons dans notre contexte de travail, est l’objet même de cette étude. Servir notre prochain est une caractéristique essentielle pour nous faire «reconnaître» dans notre identité d’enfants de Dieu. Celle-ci est incompatible avec u


---
##  2. Nettoyage du corpus

Avant toute tokenisation, le texte doit être nettoyé :
- Suppression des balises et caractères spéciaux
- Normalisation des espaces
- Filtrage des textes trop courts

In [6]:
def clean_text(text: str) -> str:
    """Nettoie un texte brut pour l'entraînement du tokenizer."""
    # Suppression des balises HTML résiduelles
    text = re.sub(r'<[^>]+>', '', text)
    # Suppression des URLs
    text = re.sub(r'http\S+|www\S+', '', text)
    # Suppression des caractères non imprimables
    text = re.sub(r'[^\x20-\x7E\u00C0-\u024F\n]', '', text)
    # Normalisation des espaces multiples
    text = re.sub(r' +', ' ', text)
    # Normalisation des sauts de ligne multiples
    text = re.sub(r'\n+', '\n', text)
    return text.strip()


print(" Nettoyage du corpus en cours...")
cleaned_texts = []
for text in tqdm(all_texts):
    cleaned = clean_text(text)
    # On garde uniquement les textes d'au moins 100 caractères
    if len(cleaned) >= 100:
        cleaned_texts.append(cleaned)

print(f" Corpus nettoyé : {len(cleaned_texts):,} documents conservés")

# Statistiques
total_chars = sum(len(t) for t in cleaned_texts)
print(f" Taille totale : {total_chars / 1e9:.2f} GB de texte")

 Nettoyage du corpus en cours...


100%|██████████| 500000/500000 [03:43<00:00, 2239.17it/s]


 Corpus nettoyé : 499,678 documents conservés
 Taille totale : 2.58 GB de texte


In [7]:
# Sauvegarde du corpus dans un fichier texte pour l'entraînement du tokenizer
corpus_path = "corpus.txt"

print(f" Sauvegarde du corpus dans {corpus_path}...")
with open(corpus_path, "w", encoding="utf-8") as f:
    for text in tqdm(cleaned_texts):
        f.write(text + "\n")

print(" Corpus sauvegardé")

 Sauvegarde du corpus dans corpus.txt...


100%|██████████| 499678/499678 [00:05<00:00, 83573.28it/s] 

 Corpus sauvegardé


---
##  3. Tokenisation — Byte Pair Encoding (BPE)

Le BPE part de caractères isolés et fusionne progressivement les paires les plus fréquentes.
On implémente d'abord le BPE **from scratch** pour comprendre la mécanique, puis on utilise la version optimisée de HuggingFace pour l'entraînement réel sur le grand corpus.

### 3a. BPE From Scratch (version pédagogique)

In [8]:
def get_vocab(corpus: list[str]) -> dict:
    """Construit le vocabulaire initial : chaque mot est découpé en caractères."""
    vocab = Counter()
    for text in corpus:
        for word in text.strip().split():
            # On ajoute un marqueur de fin de mot </w>
            word_chars = ' '.join(list(word)) + ' </w>'
            vocab[word_chars] += 1
    return dict(vocab)


def get_pairs(vocab: dict) -> Counter:
    """Compte toutes les paires de symboles adjacents dans le vocabulaire."""
    pairs = Counter()
    for word, freq in vocab.items():
        symbols = word.split()
        for i in range(len(symbols) - 1):
            pairs[(symbols[i], symbols[i+1])] += freq
    return pairs


def merge_vocab(pair: tuple, vocab: dict) -> dict:
    """Fusionne la paire la plus fréquente dans tout le vocabulaire."""
    new_vocab = {}
    bigram = ' '.join(pair)
    replacement = ''.join(pair)
    for word in vocab:
        new_word = word.replace(bigram, replacement)
        new_vocab[new_word] = vocab[word]
    return new_vocab


def train_bpe_scratch(corpus: list[str], num_merges: int = 100) -> list:
    """Entraîne le BPE from scratch sur un corpus."""
    vocab = get_vocab(corpus)
    merges = []

    print(f" Début de l'entraînement BPE ({num_merges} fusions)...")
    for i in tqdm(range(num_merges)):
        pairs = get_pairs(vocab)
        if not pairs:
            break
        # On sélectionne la paire la plus fréquente
        best_pair = max(pairs, key=pairs.get)
        vocab = merge_vocab(best_pair, vocab)
        merges.append(best_pair)

    return merges, vocab


# Démonstration sur un petit corpus
demo_corpus = [
    "Je construis un LLM from scratch",
    "Le LLM apprend à partir de données",
    "Les transformers sont puissants",
    "low lower lowest slow slower",
    "tokenisation tokenizer token",
] * 100  # On répète pour avoir des fréquences significatives

merges, final_vocab = train_bpe_scratch(demo_corpus, num_merges=50)

print(f"\n {len(merges)} fusions effectuées")
print("\n Les 10 premières fusions :")
for i, merge in enumerate(merges[:10]):
    print(f"  Fusion {i+1}: '{merge[0]}' + '{merge[1]}' → '{merge[0]+merge[1]}'")

 Début de l'entraînement BPE (50 fusions)...


100%|██████████| 50/50 [00:00<00:00, 14917.85it/s]


 50 fusions effectuées

 Les 10 premières fusions :
  Fusion 1: 's' + '</w>' → 's</w>'
  Fusion 2: 'l' + 'o' → 'lo'
  Fusion 3: 'lo' + 'w' → 'low'
  Fusion 4: 'o' + 'n' → 'on'
  Fusion 5: 'e' + 'n' → 'en'
  Fusion 6: 'r' + '</w>' → 'r</w>'
  Fusion 7: 'e' + '</w>' → 'e</w>'
  Fusion 8: 'low' + 'e' → 'lowe'
  Fusion 9: 't' + 'o' → 'to'
  Fusion 10: 'to' + 'k' → 'tok'


### 3b. BPE sur le grand corpus (version optimisée HuggingFace)

In [9]:
# Configuration du tokenizer
VOCAB_SIZE = 50_000  # GPT-2 utilise 50 257

# Initialisation du tokenizer BPE
tokenizer = Tokenizer(BPE(unk_token="[UNK]"))
tokenizer.pre_tokenizer = Whitespace()

# Configuration de l'entraîneur
trainer = BpeTrainer(
    vocab_size=VOCAB_SIZE,
    special_tokens=["[UNK]", "[CLS]", "[SEP]", "[PAD]", "[MASK]"],
    min_frequency=2,  # Un token doit apparaître au moins 2 fois
    show_progress=True
)

print(f" Entraînement du BPE sur le grand corpus...")
print(f" Taille de vocabulaire cible : {VOCAB_SIZE:,} tokens")

# Entraînement sur le fichier corpus
tokenizer.train(files=[corpus_path], trainer=trainer)

# Sauvegarde du tokenizer
tokenizer.save("tokenizer_fr.json")
print(" Tokenizer entraîné et sauvegardé dans tokenizer_fr.json")

 Entraînement du BPE sur le grand corpus...
 Taille de vocabulaire cible : 50,000 tokens



 Tokenizer entraîné et sauvegardé dans tokenizer_fr.json


In [10]:
# Test du tokenizer
test_phrases = [
    "Je construis un LLM from scratch",
    "Le mécanisme d'attention est au cœur du Transformer",
    "tokenisation tokenizer tokenizing"
]

print(" Test du tokenizer sur quelques phrases :\n")
for phrase in test_phrases:
    encoding = tokenizer.encode(phrase)
    print(f"Texte    : {phrase}")
    print(f"Tokens   : {encoding.tokens}")
    print(f"IDs      : {encoding.ids}")
    print(f"Nb tokens: {len(encoding.tokens)}")
    print("-" * 60)

 Test du tokenizer sur quelques phrases :

Texte    : Je construis un LLM from scratch
Tokens   : ['Je', 'constru', 'is', 'un', 'LL', 'M', 'from', 'sc', 'rat', 'ch']
IDs      : [729, 1095, 451, 459, 12770, 49, 5176, 1588, 2989, 470]
Nb tokens: 10
------------------------------------------------------------
Texte    : Le mécanisme d'attention est au cœur du Transformer
Tokens   : ['Le', 'mécanisme', 'd', "'", 'attention', 'est', 'au', 'cœur', 'du', 'Trans', 'former']
IDs      : [539, 10947, 72, 11, 4895, 482, 460, 3101, 477, 3218, 3369]
Nb tokens: 11
------------------------------------------------------------
Texte    : tokenisation tokenizer tokenizing
Tokens   : ['to', 'ken', 'isation', 'to', 'ken', 'i', 'zer', 'to', 'ken', 'iz', 'ing']
IDs      : [575, 7808, 2819, 575, 7808, 77, 5143, 575, 7808, 6632, 767]
Nb tokens: 11
------------------------------------------------------------


---
##  4. Token Embeddings

Chaque token reçoit un identifiant numérique. Mais un identifiant seul ne dit rien sur le sens.
On transforme chaque identifiant en un **vecteur dense** appris pendant l'entraînement.

La matrice d'embedding a une taille de : `vocab_size × embedding_dim`

In [11]:
class TokenEmbedding(nn.Module):
    """
    Matrice d'embedding : associe à chaque token ID un vecteur dense.
    
    Args:
        vocab_size    : nombre total de tokens dans le vocabulaire
        embedding_dim : dimension des vecteurs d'embedding
    """
    def __init__(self, vocab_size: int, embedding_dim: int):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        self.embedding_dim = embedding_dim

    def forward(self, token_ids: torch.Tensor) -> torch.Tensor:
        """
        Args:
            token_ids : tensor de shape (batch_size, seq_len)
        Returns:
            embeddings : tensor de shape (batch_size, seq_len, embedding_dim)
        """
        # On scale les embeddings par sqrt(d_model) comme dans le papier original
        return self.embedding(token_ids) * (self.embedding_dim ** 0.5)


# Configuration
EMBEDDING_DIM = 512  # GPT-2 small utilise 768, GPT-3 utilise 12 288

# Initialisation
token_embedding = TokenEmbedding(
    vocab_size=VOCAB_SIZE,
    embedding_dim=EMBEDDING_DIM
).to(device)

print(f" Matrice d'embedding initialisée")
print(f" Taille : {VOCAB_SIZE:,} tokens × {EMBEDDING_DIM} dimensions")
print(f" Nombre de paramètres : {VOCAB_SIZE * EMBEDDING_DIM:,}")

# Démonstration
phrase = "Je construis un LLM from scratch"
encoding = tokenizer.encode(phrase)
token_ids = torch.tensor([encoding.ids], dtype=torch.long).to(device)

embeddings = token_embedding(token_ids)
print(f"\n Phrase : '{phrase}'")
print(f" Token IDs : {encoding.ids}")
print(f" Shape des embeddings : {embeddings.shape}")
print(f"   → (batch_size=1, seq_len={len(encoding.ids)}, embedding_dim={EMBEDDING_DIM})")
print(f"\n Vecteur du premier token '{encoding.tokens[0]}' (5 premières dimensions) :")
print(f"   {embeddings[0, 0, :5].detach().cpu().numpy().round(4)}")

 Matrice d'embedding initialisée
 Taille : 50,000 tokens × 512 dimensions
 Nombre de paramètres : 25,600,000

 Phrase : 'Je construis un LLM from scratch'
 Token IDs : [729, 1095, 451, 459, 12770, 49, 5176, 1588, 2989, 470]
 Shape des embeddings : torch.Size([1, 10, 512])
   → (batch_size=1, seq_len=10, embedding_dim=512)

 Vecteur du premier token 'Je' (5 premières dimensions) :
   [ 35.9876 -35.3953  -3.2836 -23.4653  27.3556]


---
##  5. Positional Embeddings

Les token embeddings ont un angle mort : "le chat mange la souris" et "la souris mange le chat"
produisent les mêmes vecteurs. L'ordre est invisible.

On implémente les deux approches :
- **Approche fixe** : fonctions sinus/cosinus (papier original)
- **Approche apprise** : vecteurs appris comme GPT

In [12]:
class SinusoidalPositionalEmbedding(nn.Module):
    """
    Positional Embeddings fixes basés sur des fonctions sinus/cosinus.
    Approche du papier original 'Attention Is All You Need' (Vaswani et al., 2017).
    
    PE(pos, 2i)   = sin(pos / 10000^(2i/d_model))
    PE(pos, 2i+1) = cos(pos / 10000^(2i/d_model))
    """
    def __init__(self, max_seq_len: int, embedding_dim: int):
        super().__init__()
        self.embedding_dim = embedding_dim

        # Construction de la matrice de positional embeddings
        pe = torch.zeros(max_seq_len, embedding_dim)
        position = torch.arange(0, max_seq_len).unsqueeze(1).float()
        div_term = torch.exp(
            torch.arange(0, embedding_dim, 2).float() *
            (-np.log(10000.0) / embedding_dim)
        )

        # Dimensions paires → sinus
        pe[:, 0::2] = torch.sin(position * div_term)
        # Dimensions impaires → cosinus
        pe[:, 1::2] = torch.cos(position * div_term)

        # On enregistre comme buffer (pas un paramètre appris)
        self.register_buffer('pe', pe.unsqueeze(0))  # (1, max_seq_len, d_model)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Args:
            x : tensor de shape (batch_size, seq_len, embedding_dim)
        Returns:
            x + positional embeddings
        """
        seq_len = x.size(1)
        return x + self.pe[:, :seq_len, :]


class LearnedPositionalEmbedding(nn.Module):
    """
    Positional Embeddings appris — approche de GPT.
    Chaque position a son propre vecteur, appris pendant l'entraînement.
    """
    def __init__(self, max_seq_len: int, embedding_dim: int):
        super().__init__()
        self.embedding = nn.Embedding(max_seq_len, embedding_dim)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Args:
            x : tensor de shape (batch_size, seq_len, embedding_dim)
        Returns:
            x + positional embeddings
        """
        seq_len = x.size(1)
        positions = torch.arange(seq_len, device=x.device).unsqueeze(0)
        return x + self.embedding(positions)


# Configuration
MAX_SEQ_LEN = 512  # Longueur maximale de séquence

# Initialisation des deux approches
sinusoidal_pe = SinusoidalPositionalEmbedding(MAX_SEQ_LEN, EMBEDDING_DIM).to(device)
learned_pe = LearnedPositionalEmbedding(MAX_SEQ_LEN, EMBEDDING_DIM).to(device)

print(" Positional Embeddings initialisés")
print(f"   → Approche fixe (sinus/cosinus) : 0 paramètres appris")
print(f"   → Approche apprise (GPT)         : {MAX_SEQ_LEN * EMBEDDING_DIM:,} paramètres")

 Positional Embeddings initialisés
   → Approche fixe (sinus/cosinus) : 0 paramètres appris
   → Approche apprise (GPT)         : 262,144 paramètres


In [13]:
# Démonstration : même token à deux positions différentes
phrase1 = "Le chat mange la souris"
phrase2 = "La souris mange le chat"

enc1 = tokenizer.encode(phrase1)
enc2 = tokenizer.encode(phrase2)

ids1 = torch.tensor([enc1.ids], dtype=torch.long).to(device)
ids2 = torch.tensor([enc2.ids], dtype=torch.long).to(device)

# Token embeddings (sans position)
emb1 = token_embedding(ids1)
emb2 = token_embedding(ids2)

# Avec positional embeddings
emb1_with_pos = sinusoidal_pe(emb1)
emb2_with_pos = sinusoidal_pe(emb2)

print(" Démonstration : 'Le chat mange la souris' vs 'La souris mange le chat'")
print("\n SANS positional embeddings :")
print(f"   Tokens phrase 1 : {enc1.tokens}")
print(f"   Tokens phrase 2 : {enc2.tokens}")
print(f"   → Mêmes tokens dans un ordre différent = représentations différentes ? Non.")

print("\n AVEC positional embeddings (sinus/cosinus) :")
print(f"   Vecteur 'chat' en position 1 (5 dims) : {emb1_with_pos[0, 1, :5].detach().cpu().numpy().round(4)}")
print(f"   Vecteur 'chat' en position 3 (5 dims) : {emb2_with_pos[0, 3, :5].detach().cpu().numpy().round(4)}")
print(f"   → Même token, positions différentes = vecteurs différents ")

 Démonstration : 'Le chat mange la souris' vs 'La souris mange le chat'

 SANS positional embeddings :
   Tokens phrase 1 : ['Le', 'chat', 'mange', 'la', 'souris']
   Tokens phrase 2 : ['La', 'souris', 'mange', 'le', 'chat']
   → Mêmes tokens dans un ordre différent = représentations différentes ? Non.

 AVEC positional embeddings (sinus/cosinus) :
   Vecteur 'chat' en position 1 (5 dims) : [ -8.5719   0.6911 -10.6681  -4.0356  -0.4625]
   Vecteur 'chat' en position 3 (5 dims) : [12.1292 -5.4775 30.3621 22.7038 -7.4972]
   → Même token, positions différentes = vecteurs différents 


---
##  6. Pipeline complet

On assemble les 4 étapes en un seul module : **texte brut → vecteurs prêts pour le Transformer**.

In [14]:
class PreprocessingPipeline(nn.Module):
    """
    Pipeline complet de preprocessing pour un LLM.
    
    Étapes :
    1. Tokenisation (BPE)
    2. Token Embeddings
    3. Positional Embeddings
    4. Dropout (régularisation)
    """
    def __init__(
        self,
        tokenizer,
        vocab_size: int,
        embedding_dim: int,
        max_seq_len: int,
        positional_type: str = 'learned',  # 'learned' ou 'sinusoidal'
        dropout: float = 0.1
    ):
        super().__init__()
        self.tokenizer = tokenizer
        self.token_embedding = TokenEmbedding(vocab_size, embedding_dim)
        self.dropout = nn.Dropout(dropout)

        if positional_type == 'learned':
            self.positional_embedding = LearnedPositionalEmbedding(max_seq_len, embedding_dim)
        else:
            self.positional_embedding = SinusoidalPositionalEmbedding(max_seq_len, embedding_dim)

    def tokenize(self, texts: list[str]) -> torch.Tensor:
        """Tokenise une liste de textes et retourne un tensor padded."""
        encodings = [self.tokenizer.encode(text).ids for text in texts]
        max_len = max(len(e) for e in encodings)
        # Padding avec 0 (token [PAD])
        padded = [e + [0] * (max_len - len(e)) for e in encodings]
        return torch.tensor(padded, dtype=torch.long)

    def forward(self, texts: list[str]) -> torch.Tensor:
        """
        Args:
            texts : liste de textes bruts
        Returns:
            tensor de shape (batch_size, seq_len, embedding_dim)
        """
        # Étape 1 & 2 : Tokenisation → Token IDs
        token_ids = self.tokenize(texts).to(device)

        # Étape 3 : Token Embeddings
        token_emb = self.token_embedding(token_ids)

        # Étape 4 : Positional Embeddings
        output = self.positional_embedding(token_emb)

        # Dropout
        return self.dropout(output)


# Initialisation du pipeline
pipeline = PreprocessingPipeline(
    tokenizer=tokenizer,
    vocab_size=VOCAB_SIZE,
    embedding_dim=EMBEDDING_DIM,
    max_seq_len=MAX_SEQ_LEN,
    positional_type='learned',
    dropout=0.1
).to(device)

# Comptage des paramètres
total_params = sum(p.numel() for p in pipeline.parameters())
print(f" Pipeline initialisé")
print(f" Paramètres totaux : {total_params:,}")

 Pipeline initialisé
 Paramètres totaux : 25,862,144


In [15]:
# Test du pipeline complet
batch = [
    "Je construis un LLM from scratch",
    "Le mécanisme d'attention est fascinant",
    "Les transformers ont révolutionné le NLP"
]

with torch.no_grad():
    output = pipeline(batch)

print(" Pipeline complet — Résultat final :")
print(f"\n Entrée  : {len(batch)} phrases texte brut")
print(f" Sortie  : tensor de shape {tuple(output.shape)}")
print(f"   → (batch_size={output.shape[0]}, seq_len={output.shape[1]}, embedding_dim={output.shape[2]})")
print(f"\n Le preprocessing est complet.")
print(f"   Chaque token est représenté par un vecteur de {EMBEDDING_DIM} dimensions")
print(f"   portant à la fois son sens et sa position dans la séquence.")
print(f"\n  Prochaine étape : le mécanisme d'attention.")

 Pipeline complet — Résultat final :

 Entrée  : 3 phrases texte brut
 Sortie  : tensor de shape (3, 10, 512)
   → (batch_size=3, seq_len=10, embedding_dim=512)

 Le preprocessing est complet.
   Chaque token est représenté par un vecteur de 512 dimensions
   portant à la fois son sens et sa position dans la séquence.

  Prochaine étape : le mécanisme d'attention.


---
##  Récapitulatif

| Étape | Entrée | Sortie | Paramètres |
|-------|--------|--------|------------|
| Tokenisation (BPE) | Texte brut | Token IDs | 0 (algorithme) |
| Token Embeddings | Token IDs | Vecteurs (vocab_size × d_model) | 25,600,000 |
| Positional Embeddings | Vecteurs | Vecteurs + position | 262,144 |
| **Total** | **Texte brut** | **Vecteurs enrichis** | **~25.9M** |

---

**La prochaine étape : le mécanisme d'attention.**

Le modèle sait maintenant représenter chaque mot. Mais il ne sait pas encore lesquels regarder pour comprendre une phrase. C'est exactement ce que l'attention va résoudre.

---
*Série : Construire un LLM from scratch | [LinkedIn]*